### DATA PIPELINE ANTONELLA

In [34]:
import os
#test locally - PoC
# where it's running
current_dir = os.getcwd()
print(f"you are in: {current_dir}")

# folder:
audio_folder = "../raw_data/audio_and_txt_files/" 

if os.path.exists(audio_folder):
    files = [f for f in os.listdir(audio_folder) if f.endswith('.wav')]
    print(f"{len(files)} files .wav")
else:
    print("not found")

you are in: /home/totid/code/antonella-dm/projeto/notebooks
920 files .wav


# If importing from team folder instead of local
xx_folder = "../processed_data/audio_6sec/" # adjust real name

if os.path.exists(xx_folder):
    sample_file = os.listdir(xx_folder)[0]
    y, sr = librosa.load(os.path.join(xx_folder, sample_file), sr=None)
    
    print(f"check 'imported' data:")
    print(f"{len(y)/sr:.2f} s")
    print(f"sample rate: {sr} Hz")
    print(f"file qty: {len(os.listdir(xx_folder))}")

In [35]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm #shows progress
import os
import sys


#sys.path.append(os.path.abspath(os.path.join('..'))) # get m's cut
from main import main.preprocessing

In [36]:
def extract_features_raw(y, sr):
    feat = {}
    # --- TEMPORAL ---
    feat['rms_mean'] = float(np.mean(librosa.feature.rms(y=y)))
    feat['zcr_mean'] = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))
    
    # --- SPECTRAL ---
    feat['centroid_mean'] = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    feat['rolloff_mean'] = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    feat['flux_mean'] = float(np.mean(librosa.onset.onset_strength(y=y, sr=sr)))
    
    # --- MFCC (13) ---
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i, m in enumerate(mfccs):
        feat[f'mfcc_{i}'] = float(np.mean(m))
    return feat

# --- test files - 5 ---
test_files = files[:5] # only 1st 5
results = []

for f in tqdm(test_files):
    path = os.path.join(audio_folder, f)
    # Resampling para 22050Hz - 6sec - Or check if Mia has already cleaned it on her file
    y, sr = librosa.load(path, sr=22050, duration=6.0)
    
    data = extract_features_raw(y, sr)
    data['filename'] = f
    results.append(data)

# create test df
df_test = pd.DataFrame(results)
print("\n--- test result ---")
print(df_test.head())

# save locally
df_test.to_csv("teste_local_features.csv", index=False)

100%|█████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00,  9.65it/s]


--- test result ---
   rms_mean  zcr_mean  centroid_mean  rolloff_mean  flux_mean      mfcc_0  \
0  0.353043  0.000716      43.795286     41.528320   0.433962 -416.897430   
1  0.131263  0.001339     133.438929     65.929846   0.431228 -488.044525   
2  0.147832  0.023517    1299.190164   2820.226061   1.032007 -217.973022   
3  0.236638  0.007697     337.687454    289.700565   0.922199 -330.460999   
4  0.080208  0.026243    1100.981859   2400.411740   1.046043 -312.236053   

       mfcc_1     mfcc_2     mfcc_3     mfcc_4     mfcc_5     mfcc_6  \
0   62.325581  46.420395  35.231888  30.405567  29.885906  28.196938   
1   75.769440  61.811069  49.042362  39.135307  31.911039  25.999527   
2  100.233185   4.600662  29.796114  15.727752  17.164007  14.577301   
3  147.148163  79.223549  52.871487  31.837748  21.586948  15.103443   
4  141.208221   8.134517  29.663057  17.224943  30.943031  15.562644   

      mfcc_7     mfcc_8     mfcc_9    mfcc_10    mfcc_11    mfcc_12  \
0  24.402584

In [37]:
audio_folder = "../raw_data/audio_and_txt_files/" 

if os.path.exists(audio_folder):
    all_files = [f for f in os.listdir(audio_folder) if f.endswith('.wav')]
    print(f"{len(all_files)} files .wav in: {audio_folder}")
else:
    print(f"'{audio_folder}' not found")
    # interrupt exec if folder does not exist
    all_files = []

# --- extraction loop ---
if all_files:
    results = []
    
    for f in tqdm(all_files, desc="processing audiod"):
        try:
            path = os.path.join(audio_folder, f)
            
            # load:6 s, SR 22050Hz
            y, sr = librosa.load(path, sr=22050, duration=6.0)
            #use this one instead when getting the clean data (then it shoukd alredy be clean and with std Hz)
            #y, sr = librosa.load(path, sr=None)
            
            # extract (using extract_features_raw from previous cell)
            data = extract_features_raw(y, sr)
            
            # IDs so XGBoost doesnt get lost
            data['filename'] = f
            data['patient_id'] = f.split('_')[0]
            
            results.append(data)
            
        except Exception as e:
            # will let know if something is corrupted
            print(f"\n!! file {f} w error: {e}")
            continue

    # --- saving master CSV ---
    df_master = pd.DataFrame(results)
    output_folder = "../raw_data/"
    output_csv = os.path.join(output_folder, "respiratory_features_master_forXGboost.csv")
    df_master.to_csv(output_csv, index=False)
    
    print("\n" + "="*16)
    print(f"done")
    print(f"files processed: {len(df_master)}")
    print(f"CSV: {output_csv}")
    print("="*16)

920 files .wav in: ../raw_data/audio_and_txt_files/


processing audiod: 100%|██████████████████████████████████████████████████████████████| 920/920 [01:12<00:00, 12.76it/s]



done
files processed: 920
CSV: ../raw_data/respiratory_features_master_forXGboost.csv


In [38]:
import os
file_path = "../raw_data/respiratory_features_master_forXGboost.csv"
if os.path.exists(file_path):
    size = os.path.getsize(file_path) / (1024 * 1024) # size MB
    print(f"{size:.2f} MB")
    
    # lines (920)
    check_df = pd.read_csv(file_path)
    print(f"lines: {len(check_df)}")

    print('-'*10)
    print(display(check_df.head(25)))

0.32 MB
lines: 920
----------


,rms_mean,zcr_mean,centroid_mean,rolloff_mean,flux_mean,mfcc_0,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,filename,patient_id
0,0.353043,0.000716,43.795286,41.528320,0.433962,-416.897430,62.325581,46.420395,35.231888,30.405567,29.885906,28.196938,24.402584,21.075634,19.104095,17.423349,14.515306,10.367791,223_1b1_Pr_sc_Meditron.wav,223
1,0.131263,0.001339,133.438929,65.929846,0.431228,-488.044525,75.769440,61.811069,49.042362,39.135307,31.911039,25.999527,21.384661,17.560677,14.248250,11.581615,10.005250,9.039628,134_2b2_Al_mc_LittC2SE.wav,134
2,0.147832,0.023517,1299.190164,2820.226061,1.032007,-217.973022,100.233185,4.600662,29.796114,15.727752,17.164007,14.577301,18.850311,10.304487,13.588282,5.419698,6.978378,4.364567,170_1b4_Al_mc_AKGC417L.wav,170
3,0.236638,0.007697,337.687454,289.700565,0.922199,-330.460999,147.148163,79.223549,52.871487,31.837748,21.586948,15.103443,11.056579,8.015654,6.217966,4.712320,3.982452,3.184886,203_1p3_Pl_mc_AKGC417L.wav,203
4,0.080208,0.026243,1100.981859,2400.411740,1.046043,-312.236053,141.208221,8.134517,29.663057,17.224943,30.943031,15.562644,-0.528191,-3.173283,9.954062,7.148951,6.205405,1.809195,207_2b3_Tc_mc_AKGC417L.wav,207
5,0.035995,0.009579,451.795425,930.666702,1.082076,-451.197388,181.109467,9.645131,-18.176332,18.386055,22.060270,5.537070,3.691941,2.975734,11.476917,18.128443,15.646124,2.962984,188_1b1_Tc_sc_Meditron.wav,188
6,0.102974,0.009556,508.360669,583.350269,1.110870,-351.141235,153.818405,39.326012,22.173399,24.587170,25.988836,17.501591,12.140687,10.192830,7.306312,5.662916,8.342436,7.403780,170_2b2_Tc_mc_AKGC417L.wav,170
7,0.265474,0.001380,65.981993,76.239179,0.529541,-391.806763,116.913414,61.570641,23.370325,16.057096,21.499102,22.249159,19.670801,20.261456,21.551264,18.241104,11.879057,6.945556,157_1b1_Pl_sc_Meditron.wav,157
8,0.081462,0.008431,622.671776,690.683726,0.940197,-350.903839,140.261200,50.939663,30.961023,16.777407,17.194637,10.012837,9.698359,4.820922,5.891583,3.171983,5.926676,5.099188,177_2b4_Pl_mc_AKGC417L.wav,177
9,0.036550,0.005431,262.414257,221.151816,0.651884,-510.306824,134.938675,73.466927,26.575851,11.914156,15.869065,19.255404,15.428850,9.519996,7.766943,10.220729,12.608171,12.133439,156_2b3_Lr_mc_AKGC417L.wav,156


None


In [39]:
print(f"last loaded signal: {y.shape}") 
# for SR=22050 and 6s, shape must be 132300 (22050 * 6)

last loaded signal: (132300,)
